# Credit risk project: data exploration

**Aim:** build loan-level PD and LGD models on the Lending Club accepted-loans file, then a portfolio loss simulation and stress tests.


**Sections**
1. Loading the data
2. Loan status
3. Issue dates
4. Column selection
5. Missing values and data types
6. Summary of decisions

## 1. Load data

In [1]:
import sys
sys.path.insert(0, "../src")
import pandas as pd

# show every row and column, no truncation
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

DATA_PATH = "../data/raw/accepted_2007_to_2018Q4.csv.gz"

In [2]:
# all rows, all columns
df = pd.read_csv(DATA_PATH, low_memory=False)

print(df.shape)
print(f"memory: {df.memory_usage(deep=True).sum() / 1e9:.1f} GB")

(2260701, 151)
memory: 6.3 GB


In [3]:
# how many columns of each type
print(df.dtypes.astype(str).value_counts())

float64    113
str         38
Name: count, dtype: int64


In [4]:
# first rows, flipped so each column is a row
df.head(3).T

,0,1,2
id,68407277,68355089,68341763
member_id,NaN,NaN,NaN
loan_amnt,3600.0,24700.0,20000.0
funded_amnt,3600.0,24700.0,20000.0
funded_amnt_inv,3600.0,24700.0,20000.0
term,36 months,36 months,60 months
int_rate,13.99,11.99,10.78
installment,123.03,820.28,432.66
grade,C,C,B
sub_grade,C4,C1,B4


In [5]:
# last rows, in case the file ends with something that isn't a loan
df.tail(3).T

,2260698,2260699,2260700
id,88215728,Total amount funded in policy code 1: 1465324575,Total amount funded in policy code 2: 521953170
member_id,NaN,NaN,NaN
loan_amnt,14000.0,NaN,NaN
funded_amnt,14000.0,NaN,NaN
funded_amnt_inv,14000.0,NaN,NaN
term,60 months,NaN,NaN
int_rate,14.49,NaN,NaN
installment,329.33,NaN,NaN
grade,C,NaN,NaN
sub_grade,C4,NaN,NaN


**What we see**
- 2,260,701 rows, 151 columns (113 numeric, 38 text), 6.3 GB.
- Each row is a loan: its terms, the borrower, and a `loan_status`.
- Some columns record what happened after issue, e.g. `last_pymnt_d`, payment totals, `recoveries`.
- The last two rows are not loans: `id` holds a text total and every other column is empty.

**Questions**
1. Which rows are not loans?
2. When was the snapshot taken? It matters for which loans have had time to finish.


In [6]:
# rows with no status or no issue date
not_loans = df[df["loan_status"].isna() | df["issue_d"].isna()]
print(len(not_loans))
not_loans[["id", "loan_amnt", "loan_status", "issue_d"]]

33


,id,loan_amnt,loan_status,issue_d
421095,Total amount funded in policy code 1: 6417608175,NaN,NaN,NaN
421096,Total amount funded in policy code 2: 1944088810,NaN,NaN,NaN
528961,Total amount funded in policy code 1: 1741781700,NaN,NaN,NaN
528962,Total amount funded in policy code 2: 564202131,NaN,NaN,NaN
651664,Total amount funded in policy code 1: 1791201400,NaN,NaN,NaN
651665,Total amount funded in policy code 2: 651669342,NaN,NaN,NaN
749520,Total amount funded in policy code 1: 1443412975,NaN,NaN,NaN
749521,Total amount funded in policy code 2: 511988838,NaN,NaN,NaN
877716,Total amount funded in policy code 1: 2063142975,NaN,NaN,NaN
877717,Total amount funded in policy code 2: 823319310,NaN,NaN,NaN


In [7]:
# latest date in each date column
for col in ["issue_d", "last_pymnt_d", "last_credit_pull_d", "next_pymnt_d"]:
    print(col, pd.to_datetime(df[col], format="%b-%Y").max())

issue_d 2018-12-01 00:00:00
last_pymnt_d 2019-03-01 00:00:00
last_credit_pull_d 2019-04-01 00:00:00
next_pymnt_d 2019-05-01 00:00:00



- 33 rows are not loans: 32 "Total amount funded" footers and 1 "Loans that do not meet the credit policy" header.
- Last issue date is Dec-2018. Last payment is Mar-2019 and last credit pull is Apr-2019, so the snapshot is around Mar-Apr 2019.

**Decision**
Drop the 33 non-loan rows. 

In [8]:
# drop the 33 rows that are not loans
df = df.drop(index=not_loans.index)
print(df.shape)

(2260668, 151)


In [9]:
# term values have a leading space, e.g. " 36 months"
df["term"] = df["term"].str.strip()

## 2. Loan status

**Question:** which loans have a known outcome we can label?

In [10]:
df["loan_status"].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

In [11]:
# outstanding principal still owed, by status
df.groupby("loan_status")["out_prncp"].agg(["count", "mean", "median"]).round(0)

,count,mean,median
loan_status,,,
Charged Off,268559,0.0,0.0
Current,878317,10390.0,8316.0
Default,40,9573.0,7711.0
Does not meet the credit policy. Status:Charged Off,761,0.0,0.0
Does not meet the credit policy. Status:Fully Paid,1988,0.0,0.0
Fully Paid,1076751,0.0,0.0
In Grace Period,8436,10911.0,8936.0
Late (16-30 days),4349,11025.0,9017.0
Late (31-120 days),21467,11378.0,9313.0


**Observations**
- Fully Paid, Charged Off and the two "Does not meet the credit policy" statuses have no principal outstanding, so those loans are finished.
- Current, In Grace Period and Late loans still owe about $10-11k on average, so their outcome isn't known yet.
- `Default` (40 loans) still owes about $9.6k, like the open loans, but the borrower has already defaulted.

**Decision**
- `default = 1`: Charged Off, Default, and the legacy Charged Off label
- `default = 0`: Fully Paid and the legacy Fully Paid label
- Current, In Grace Period and Late are excluded. Their outcome is unknown, so any label would be a guess.

In [12]:
final_statuses = [ 
    "Fully Paid",
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Fully Paid",
    "Does not meet the credit policy. Status:Charged Off",
]
default_statuses = [  
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Charged Off",
]

labelled = df[df["loan_status"].isin(final_statuses)].copy()
labelled["default"] = labelled["loan_status"].isin(default_statuses).astype(int)

print(len(labelled), labelled["default"].mean())
print(labelled["default"].value_counts())

1348099 0.19980728418313493
default
0    1078739
1     269360
Name: count, dtype: int64


## 3. Issue dates

**Question:** which loans were issued when, and for which years can we trust the labels?

In [13]:
# issue year, added to both the full data and the labelled loans
df["issue_year"] = pd.to_datetime(df["issue_d"], format="%b-%Y").dt.year
labelled["issue_year"] = pd.to_datetime(labelled["issue_d"], format="%b-%Y").dt.year

df["issue_year"].value_counts().sort_index()

/var/folders/1q/2rl244c9583f9rhcm7rn3r_r0000gn/T/ipykernel_17855/4153589460.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["issue_year"] = pd.to_datetime(df["issue_d"], format="%b-%Y").dt.year


issue_year
2007       603
2008      2393
2009      5281
2010     12537
2011     21721
2012     53367
2013    134814
2014    235629
2015    421095
2016    434407
2017    443579
2018    495242
Name: count, dtype: int64

In [14]:
# share of loans with a known outcome, by issue year and term
(labelled.groupby(["issue_year", "term"]).size()
 / df.groupby(["issue_year", "term"]).size()).unstack().round(3)

term,36 months,60 months
issue_year,,
2007,1.000,NaN
2008,1.000,NaN
2009,1.000,NaN
2010,1.000,1.000
2011,1.000,1.000
2012,1.000,1.000
2013,1.000,1.000
2014,1.000,0.829
2015,0.999,0.671


In [15]:
# loans with no known outcome yet
open_statuses = ["Current", "In Grace Period", "Late (16-30 days)", "Late (31-120 days)"]
open_loans = df[df["loan_status"].isin(open_statuses)]

pd.crosstab([open_loans["issue_year"], open_loans["term"]], open_loans["loan_status"])

loan_status           Current  In Grace Period  Late (16-30 days)  Late (31-120 days)
issue_year term                                                                      
2013       60 months        6                1                  0                   3
2014       60 months    11919              200                 73                 334
2015       36 months       72                9                  5                  61
           60 months    43227              603                274                1298
2016       36 months    86623             1096                584                2831
           60 months    47438              697                318                1715
2017       36 months   183950             1860                957                5101
           60 months    77901             1051                535                2903
2018       36 months   296050             1869                999                4481
           60 months   131131             1050                604                2740

In [16]:
# default rate among labelled loans, by issue year and term
labelled.groupby(["issue_year", "term"])["default"].agg(["count", "mean"]).unstack().round(3)

count                mean          
term       36 months 60 months 36 months 60 months
issue_year                                        
2007           603.0       NaN     0.262       NaN
2008          2393.0       NaN     0.207       NaN
2009          5281.0       NaN     0.137       NaN
2010          9156.0    3381.0     0.109     0.224
2011         14101.0    7620.0     0.106     0.236
2012         43470.0    9897.0     0.136     0.277
2013        100422.0   34382.0     0.123     0.251
2014        162570.0   60533.0     0.137     0.311
2015        283026.0   92520.0     0.149     0.364
2016        232361.0   60744.0     0.198     0.364
2017        128551.0   40770.0     0.200     0.330
2018         41272.0   15046.0     0.132     0.226

In [17]:
# last year with (almost) every loan labelled, per term
last_clean_year = {"36 months": 2015, "60 months": 2013}

in_window = labelled["issue_year"] <= labelled["term"].map(last_clean_year)
usable = labelled[in_window].copy() # completed loans within selected window that will be used for further analysis

print(len(usable))
usable.groupby(["issue_year", "term"]).size().unstack()

676302


term,36 months,60 months
issue_year,,
2007,603.0,NaN
2008,2393.0,NaN
2009,5281.0,NaN
2010,9156.0,3381.0
2011,14101.0,7620.0
2012,43470.0,9897.0
2013,100422.0,34382.0
2014,162570.0,NaN
2015,283026.0,NaN


In [18]:
# open loans that fall inside the window
open_in_window = open_loans[open_loans["issue_year"] <= open_loans["term"].map(last_clean_year)]
print(len(open_in_window))
open_in_window["loan_status"].value_counts()

157


loan_status
Current               78
Late (31-120 days)    64
In Grace Period       10
Late (16-30 days)      5
Name: count, dtype: int64

**Decision**
Keep loans issued up to 2015 (36-month) and up to 2013 (60-month): 676,302 labelled loans. The 157 unlabelled loans inside this window are dropped.

**Why**
- Recent loans haven't had time to finish, so the labelled ones are a biased sample.
- Share labelled: 36-month loans reach 99.9% in 2015 and drop to 72% in 2016. 60-month loans are fully labelled through 2013 and 83% in 2014.
- Past these years the labelled share is small (2018, 36-month: 12%), so the 13% default rate among those loans isn't comparable with finished vintages.

**Limitation**
- The target is the final outcome of a finished loan, not a fixed-horizon default like a 12-month PD.
- No 60-month vintage after 2013 is fully labelled, so the out-of-time test set will be 36-month only.

The first year is decided after the column review (section 4).

## 4. Column selection

**Question:** which columns can a PD model use?

A PD model scores a loan when it is issued, so it can only use what is known at that point.

### 4.1 Columns recorded after issue

A PD model scores a loan when it is issued. Columns that record payments, collections or later events can't be used.

In [19]:
payment_cols = ["out_prncp", "out_prncp_inv", "total_pymnt", "total_pymnt_inv", "total_rec_prncp",
                "total_rec_int", "total_rec_late_fee", "recoveries", "collection_recovery_fee",
                "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d", "last_credit_pull_d",
                "last_fico_range_high", "last_fico_range_low"]

hardship_cols = ["hardship_flag", "hardship_type", "hardship_reason", "hardship_status", "deferral_term",
                 "hardship_amount", "hardship_start_date", "hardship_end_date", "payment_plan_start_date",
                 "hardship_length", "hardship_dpd", "hardship_loan_status",
                 "orig_projected_additional_accrued_interest", "hardship_payoff_balance_amount",
                 "hardship_last_payment_amount"]

settlement_cols = ["debt_settlement_flag", "debt_settlement_flag_date", "settlement_status",
                   "settlement_date", "settlement_amount", "settlement_percentage", "settlement_term"]

leakage_cols = payment_cols + hardship_cols + settlement_cols
usable = usable.drop(columns=leakage_cols)
print(len(leakage_cols), usable.shape)

37 (676302, 116)


### 4.2 Administrative columns

Links, free text and platform fields. They say nothing about the borrower's risk.

In [20]:
admin_cols = ["url", "desc", "title", "emp_title", "zip_code", "policy_code",
              "pymnt_plan", "initial_list_status", "disbursement_method", "funded_amnt_inv"]

usable = usable.drop(columns=admin_cols)
print(len(admin_cols), usable.shape)

10 (676302, 106)


### 4.3 Missing Columns

In [21]:
# % missing per column, by issue year, for the labelled loans in the window (rows = columns)
missing_by_year = (usable.drop(columns=["issue_year", "default"])
                   .isnull()
                   .groupby(usable["issue_year"])
                   .mean()
                   .mul(100).round(1)
                   .T)

missing_by_year

issue_year,2007,2008,2009,2010,2011,2012,2013,2014,2015
id,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
member_id,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
loan_amnt,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
funded_amnt,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
term,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
int_rate,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
installment,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
grade,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
sub_grade,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
emp_length,0.0,0.0,0.0,2.8,3.5,3.6,4.4,5.9,6.7


In [22]:
# columns still in the data that are at least 90% empty in every issue year
m = missing_by_year.loc[missing_by_year.index.isin(usable.columns)]
empty_cols = m.index[m.min(axis=1) >= 90].tolist()

print(len(empty_cols), "columns")
m.loc[empty_cols]

31 columns


issue_year,2007,2008,2009,2010,2011,2012,2013,2014,2015
member_id,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
annual_inc_joint,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,99.9
dti_joint,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,99.9
verification_status_joint,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,99.9
open_acc_6m,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,94.7
open_act_il,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,94.7
open_il_12m,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,94.7
open_il_24m,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,94.7
mths_since_rcnt_il,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,94.9
total_bal_il,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,94.7


In [23]:
usable = usable.drop(columns=empty_cols)
print(len(empty_cols), usable.shape)

31 (676302, 75)




**Decision**
Drop the 31 columns that are at least 90% empty in every issue year: `member_id`, the joint-applicant fields, and 14 account-activity fields.


### 4.4 columns that don't exist in the early years

In [24]:
# columns still in the data that are at least 90% , by year
m = missing_by_year.loc[missing_by_year.index.isin(usable.columns)]
late_cols = m.index[m[2009] >= 90].tolist()

print(len(late_cols), "columns")
m.loc[late_cols]

38 columns


issue_year,2007,2008,2009,2010,2011,2012,2013,2014,2015
mths_since_last_record,7.0,60.6,95.2,94.0,94.7,97.2,88.1,81.1,81.3
mths_since_last_major_derog,100.0,100.0,100.0,100.0,100.0,89.9,79.9,70.7,69.8
tot_coll_amt,100.0,100.0,100.0,100.0,100.0,52.0,0.0,0.0,0.0
tot_cur_bal,100.0,100.0,100.0,100.0,100.0,52.0,0.0,0.0,0.0
total_rev_hi_lim,100.0,100.0,100.0,100.0,100.0,52.0,0.0,0.0,0.0
acc_open_past_24mths,100.0,100.0,100.0,100.0,100.0,14.0,0.0,0.0,0.0
avg_cur_bal,100.0,100.0,100.0,100.0,100.0,52.0,0.0,0.0,0.0
bc_open_to_buy,100.0,100.0,100.0,100.0,100.0,15.0,0.7,1.1,1.0
bc_util,100.0,100.0,100.0,100.0,100.0,15.1,0.8,1.2,1.1
mo_sin_old_il_acct,100.0,100.0,100.0,100.0,100.0,54.2,3.7,3.6,3.5


In [25]:
print(usable.shape)   # expect (676302, 75)

# total % missing per column, highest first
pd.DataFrame({"% missing": usable.isnull().mean().mul(100).round(1)}).sort_values("% missing", ascending=False)

(676302, 75)


,% missing
mths_since_last_record,84.5
mths_since_recent_bc_dlq,77.2
mths_since_last_major_derog,75.5
mths_since_recent_revol_delinq,67.9
mths_since_last_delinq,51.7
mths_since_recent_inq,17.3
mo_sin_old_il_acct,13.6
num_tl_120dpd_2m,13.5
tot_cur_bal,10.4
tot_coll_amt,10.4


### 4.5 High-missing columns

**Decision**
Drop the 4 columns that are more than 50% missing (68-85%): `mths_since_last_record`, `mths_since_recent_bc_dlq`, `mths_since_last_major_derog`, `mths_since_recent_revol_delinq`. Keep `mths_since_last_delinq` (52% missing), because a missing value there means the borrower has never been delinquent. All other columns are at most 17% missing and get imputed.

In [26]:
# columns that are at least 90% before 2012
m = missing_by_year.loc[missing_by_year.index.isin(usable.columns)]
late_cols = m.index[m[2009] >= 90].tolist()

print(usable.shape)   # expect (676302, 75)
print(len(late_cols), "columns")
m.loc[late_cols]      # % empty by issue year for each one

(676302, 75)
38 columns


issue_year,2007,2008,2009,2010,2011,2012,2013,2014,2015
mths_since_last_record,7.0,60.6,95.2,94.0,94.7,97.2,88.1,81.1,81.3
mths_since_last_major_derog,100.0,100.0,100.0,100.0,100.0,89.9,79.9,70.7,69.8
tot_coll_amt,100.0,100.0,100.0,100.0,100.0,52.0,0.0,0.0,0.0
tot_cur_bal,100.0,100.0,100.0,100.0,100.0,52.0,0.0,0.0,0.0
total_rev_hi_lim,100.0,100.0,100.0,100.0,100.0,52.0,0.0,0.0,0.0
acc_open_past_24mths,100.0,100.0,100.0,100.0,100.0,14.0,0.0,0.0,0.0
avg_cur_bal,100.0,100.0,100.0,100.0,100.0,52.0,0.0,0.0,0.0
bc_open_to_buy,100.0,100.0,100.0,100.0,100.0,15.0,0.7,1.1,1.0
bc_util,100.0,100.0,100.0,100.0,100.0,15.1,0.8,1.2,1.1
mo_sin_old_il_acct,100.0,100.0,100.0,100.0,100.0,54.2,3.7,3.6,3.5


In [27]:
usable = usable.drop(columns=late_cols)
print(usable.shape)

(676302, 37)


**Decision**
Drop 38 columns: 34 credit-detail fields and 4 `mths_since_*` fields. Keep every issue year, 2007-2015: 676,302 loans, 32 features. `mths_since_last_delinq` stays, because a missing value there means the borrower has never been delinquent.

**Why**
- The 34 credit-detail fields are 100% empty through 2011 and 14-54% empty in 2012. Imputing them would fill whole years with one number.
- The 4 `mths_since_*` fields are 68-85% empty overall.

**To compare later**
Start at 2013 and keep the 34 credit-detail fields. The dropped columns are still in `labelled`, so this version can be rebuilt.

In [28]:
# % missing by issue year for every column that remains (rows = columns)
remaining = usable.drop(columns=["issue_year", "default"])
by_year = remaining.isnull().groupby(usable["issue_year"]).mean().mul(100).T
by_year["overall"] = remaining.isnull().mean().mul(100)

print(usable.shape)   # expect (676302, 37)
by_year.round(1)

(676302, 37)


issue_year,2007,2008,2009,2010,2011,2012,2013,2014,2015,overall
id,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
loan_amnt,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
funded_amnt,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
term,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
int_rate,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
installment,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
grade,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
sub_grade,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
emp_length,0.0,0.0,0.0,2.8,3.5,3.6,4.4,5.9,6.7,5.6
home_ownership,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 4.5 Redundant columns

**Decision**
Drop `funded_amnt` and keep `loan_amnt`. 31 features remain.

**Why**
`funded_amnt` equals `loan_amnt` for 99.7% of loans, and almost all the differences are in 2007-2011.

**Kept for now**
- `fico_range_low` and `fico_range_high` are 4 apart for all but 73 loans, so they carry almost the same information.
- `application_type` has Joint App loans only in 2015 (239 loans), so in 2007-2014 every loan is Individual.

In [29]:
# Is funded_amnt a copy of loan_amnt?
print((usable["loan_amnt"] == usable["funded_amnt"]).mean())
(usable["loan_amnt"] != usable["funded_amnt"]).groupby(usable["issue_year"]).mean().round(4)

0.9969510662396385


issue_year
2007    0.0381
2008    0.1270
2009    0.0057
2010    0.0814
2011    0.0285
2012    0.0011
2013    0.0001
2014    0.0000
2015    0.0000
dtype: float64

In [30]:
# Is the FICO pair one piece of information? If the gap is always 4, we merge it into one column.
(usable["fico_range_high"] - usable["fico_range_low"]).value_counts()

4.0    676229
5.0        73
Name: count, dtype: int64

In [31]:
# Does application_type vary in the training years?
pd.crosstab(usable["issue_year"], usable["application_type"])

application_type,Individual,Joint App
issue_year,,
2007,603,0
2008,2393,0
2009,5281,0
2010,12537,0
2011,21721,0
2012,53367,0
2013,134804,0
2014,162570,0
2015,282787,239


In [32]:
# funded_amnt equals loan_amnt for 99.7% of loans, so keep loan_amnt only
usable = usable.drop(columns=["funded_amnt"])
print(usable.shape)

(676302, 36)


In [33]:
# term: what values does it hold?
print(usable["term"].unique())

# emp_length: every value and how often it appears
print(usable["emp_length"].value_counts(dropna=False))

# credit history length in years at issue (a negative minimum would mean a bad parse)
hist = (pd.to_datetime(usable["issue_d"], format="%b-%Y")
        - pd.to_datetime(usable["earliest_cr_line"], format="%b-%Y")).dt.days / 365.25
hist.describe().round(1)

<StringArray>
['36 months', '60 months']
Length: 2, dtype: str
emp_length
10+ years    211357
2 years       61436
< 1 year      55418
3 years       54531
1 year        44832
5 years       43760
4 years       40910
NaN           37605
7 years       34187
6 years       33676
8 years       32940
9 years       25650
Name: count, dtype: int64


count    676273.0
mean         16.1
std           7.5
min           0.5
25%          11.0
50%          14.6
75%          19.8
max          71.0
dtype: float64

### 4.6 Text columns

**Decision**
- `term` becomes 36 or 60.
- `emp_length` becomes 0-10 (`< 1 year` is 0, `10+ years` is 10). The 37,605 missing values (5.6%) stay missing.
- `earliest_cr_line` is replaced by `credit_history_years`, the credit history length at issue. 29 loans have none, all from 2007. these are handles in data_prep.py
- The other text columns (`grade`, `sub_grade`, `home_ownership`, `verification_status`, `purpose`, `addr_state`, `application_type`) are categories, handled in the WoE notebook.

In [34]:
non_features = ["id", "issue_d", "loan_status", "issue_year", "default"]
final_features = [c for c in usable.columns if c not in non_features]

print(len(final_features))
final_features

31


['loan_amnt',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'purpose',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'collections_12_mths_ex_med',
 'application_type',
 'acc_now_delinq',
 'chargeoff_within_12_mths',
 'delinq_amnt',
 'pub_rec_bankruptcies',
 'tax_liens']

In [35]:
# Check data_prep.py 

import credit_risk.data_prep as dp

check = dp.restrict_vintages(dp.build_resolved(dp.load_needed(DATA_PATH)))
print(check.shape)                  # expect (676302, 41)
print(len(dp.FEATURE_COLS))         # expect 31

# same loans as this notebook
print(check["id"].sort_values().equals(usable["id"].sort_values()))

# same features as this notebook
print(set(dp.FEATURE_COLS) == set(usable.columns) - set(non_features))

X, y = dp.get_X_y(dp.clean_types(check))
print(X.shape, round(y.mean(), 3))  # expect (676302, 31) and about 0.149

(676302, 41)
31
True
True
(676302, 31) 0.149
